# Memory in Agentic Systems, with the NeMo Agent Toolkit

This notebook builds on `01_nat_basics.ipynb`. It covers two things:

1. **Concepts**: what "memory" means for an agent, and why it's different from both a chat history and a RAG
   knowledge base.
2. **Practice**: how the toolkit's memory module actually works — the `memory:` config section, the built-in
   `add_memory` / `get_memory` / `delete_memory` tools, and a hands-on example using a self-hosted Redis backend.

**What we'll do**
1. Working memory vs. long-term memory, and memory vs. RAG
2. The toolkit's memory architecture (`MemoryEditor`, the `memory:` section, the memory tools)
3. Survey the memory backends the toolkit ships, and why we'll use Redis
4. Start a local Redis Stack container
5. Configure a memory-enabled agent
6. Teach the agent a fact, then recall it in a *separate* process — proving it's real persistence, not just
   conversation context
7. Clean up (`delete_memory`, stop the container)

## 1. Concepts: what is "memory" for an agent?

It's easy to conflate three different things that all look like "the agent remembers something":

| | Lives where | Survives... | Example in this repo |
|---|---|---|---|
| **Working memory** | The prompt / context window | ...the current run, not a new session | `max_history` and the ReAct scratchpad in `01_nat_basics.ipynb` |
| **RAG / knowledge base** | An external, curated document store | ...forever, but the agent doesn't write to it | `webpage_query`, `document_search` (mentioned in notebook 1) |
| **Long-term memory** | An external store the agent itself writes to | ...new sessions, new processes, new days | This notebook |

The distinguishing feature of *memory* (as opposed to RAG) is that the agent is both the writer and the reader: it
decides what's worth remembering about a specific user, saves it, and later retrieves it — usually by semantic
(embedding) similarity rather than exact keyword match, since "what does this user usually order" needs to match
against "I'll have the usual" without any literal word overlap.

A couple of other distinctions worth knowing, even though this notebook's hands-on example won't exercise all of
them:

- **Semantic memory** (facts, preferences — "is vegetarian", "prefers Rust") vs. **episodic memory** (specific past
  events — "complained about a late delivery on March 3rd"). Most agent memory tools store both as free-text
  "memory" strings and don't distinguish between them structurally.
- Memory is **scoped**, almost always by a `user_id` (and sometimes a `conversation_id`) — it's not one global pool
  shared across every user of the agent.
- Real design problems you'll hit past the demo stage: unbounded growth (old memories need pruning or
  summarization), staleness/contradiction ("moved to Seattle" should supersede "lives in Austin", not just add to
  it), retrieval precision (`top_k` too low misses relevant facts, too high drowns the prompt in noise), and
  privacy (you're persisting personal data about a real person across sessions — that needs consent and a deletion
  path, which is exactly why `delete_memory` exists as a first-class tool, not an afterthought).

## 2. How the toolkit implements this

Verified directly against the installed `nvidia-nat==1.8.0` source (`nat/tool/memory_tools/`, `nat/memory/models.py`):

- A top-level **`memory:`** config section defines one or more named memory *client* instances (analogous to how
  `llms:` defines named LLM instances). Each backend implements the same abstract interface
  (`add_items(...)` / `search(query, top_k, user_id)` / `remove_items(user_id)`), so the tools below don't care
  which backend is behind them.
- Three built-in **functions** wrap that interface so an agent can call it like any other tool:
  - **`add_memory`** — takes a `MemoryItem` (`conversation` or a plain `memory` string, plus a required `user_id`,
    optional `tags`/`metadata`) and stores it.
  - **`get_memory`** — takes a `SearchMemoryInput` (`query`, `top_k`, `user_id`) and returns the most semantically
    similar memories for that user as JSON.
  - **`delete_memory`** — takes a `user_id` and removes everything stored for that user.
- Each of these functions takes a `memory:` field naming *which* memory client instance to use — so you add them to
  a workflow exactly like any other tool: put them in `functions:`, then list their names in
  `workflow.tool_names`.

One practical detail that matters for the demo below: `user_id` is a **required field the LLM fills in** when it
calls `add_memory`/`get_memory`/`delete_memory` — it is not silently injected for you. `nat run` does have a
`--user_id` flag, but that's for session-level scoping in the runtime, not for auto-populating tool-call arguments.
So our prompts below explicitly state the user's ID, to make sure the agent uses the *same* one on every call.

## 3. Which memory backend to use

The toolkit ships four memory backend plugins. We checked each one's source directly rather than trust the docs
(after being burned by doc/release drift twice in `02_prompt_optimization.ipynb`):

| Backend | Package | What it needs |
|---|---|---|
| **`redis_memory`** | `nvidia-nat-redis` | A **self-hosted, free** Redis Stack server (RediSearch + RedisJSON) — what we use below |
| `mem0_memory` | `nvidia-nat-mem0ai` | A **Mem0 Cloud** account (`MEM0_API_KEY`) — the installed plugin only wraps Mem0's cloud `AsyncMemoryClient`, not its local/OSS mode |
| `zep_cloud` | `nvidia-nat-zep-cloud` | A **Zep Cloud** account (API key) |
| `memmachine` | `nvidia-nat-memmachine` | A running **MemMachine** service |

Redis is the only one of the four that doesn't require signing up for a third-party service — it just needs a
container you run yourself, which keeps this notebook in the same spirit as notebooks 1 and 2 (no accounts besides
your NVIDIA API key). That's the only reason we picked it; the config pattern (`memory:` + `add_memory`/`get_memory`)
is identical for any of the four.

## 4. Setup

In [ ]:
# Only needed if you didn't install requirements.txt into this kernel's environment already.
# The "redis" extra pulls in the redis-py client used by the redis_memory backend.
# %pip install -q "nvidia-nat[langchain,redis]==1.8.0"

In [ ]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA_API_KEY (from build.nvidia.com): ")
else:
    print("NVIDIA_API_KEY already set.")

### Start a local Redis Stack container

`redis_memory` uses RediSearch (for the vector index) and RedisJSON (to store structured memory items), so it
needs the **Redis Stack** image, not plain `redis`. This requires Docker (or Podman). If you don't have Docker,
install `redis-stack-server` some other way (e.g. `brew install redis-stack` on macOS) and make sure it's listening
on `localhost:6379` before continuing — or read the rest of this notebook without running the memory cells; the
concepts and config still stand on their own.

In [ ]:
!docker run -d --name nat-memory-redis -p 6379:6379 redis/redis-stack-server:latest

In [ ]:
# Sanity check before we spend API calls — fail fast with a clear message if Redis isn't reachable.
import redis

try:
    redis.Redis(host="localhost", port=6379, socket_connect_timeout=3).ping()
    print("Redis is reachable on localhost:6379.")
except redis.exceptions.ConnectionError as e:
    print("Could not reach Redis on localhost:6379 — start the container above (or your own Redis Stack) "
          "before running the cells below.")
    raise

## 5. Configure a memory-enabled agent

New sections compared to notebook 1's `workflow.yml`:

- **`embedders:`** — `redis_memory` needs to embed each memory for semantic search, so we point it at an NVIDIA
  NIM embedding model (the same pattern used for RAG-style tools). Both this and `nim_llm` below use `_type: nim`
  rather than the generic `_type: openai` — `langchain_nvidia_ai_endpoints`'s clients know how to correctly handle
  NIM-specific request extensions (the embedding model's required `input_type` parameter, and `nim_llm`'s
  `chat_template_kwargs`) that the generic OpenAI-compatible client doesn't know exist and mishandles. `base_url`
  is driven by an optional `NVIDIA_BASE_URL` environment variable — see the README for details.
- **`memory:`** — names our Redis-backed memory client `user_memory`.
- **`functions.add_memory` / `functions.get_memory`** — both reference `memory: user_memory`, and are then added to
  `workflow.tool_names` just like `current_datetime`.

In [ ]:
import yaml

# base_url is driven by an optional NVIDIA_BASE_URL environment variable — see the README.
NIM_BASE_URL = "${NVIDIA_BASE_URL}"


def build_memory_config(include_delete=False):
    """Build the memory-enabled workflow config as a dict, so the two YAML variants below (with
    and without delete_memory) can never drift out of sync with each other again."""
    tool_names = ["add_memory", "get_memory", "current_datetime"]
    functions = {
        "current_datetime": {"_type": "current_datetime"},
        "add_memory": {"_type": "add_memory", "memory": "user_memory"},
        "get_memory": {"_type": "get_memory", "memory": "user_memory"},
    }
    additional_instructions = (
        "Whenever the user tells you their user ID, use that exact string as the user_id "
        "argument for the add_memory and get_memory tools. Proactively call add_memory when the "
        "user shares a stable fact or preference about themselves, and call get_memory before "
        "answering a question that depends on something you might already know about them.")

    if include_delete:
        tool_names.insert(2, "delete_memory")
        functions["delete_memory"] = {"_type": "delete_memory", "memory": "user_memory"}
        additional_instructions = (
            "Whenever the user tells you their user ID, use that exact string as the user_id "
            "argument for the add_memory, get_memory, and delete_memory tools.")

    return {
        # Both the embedder and the LLM use _type: nim (not _type: openai): langchain_nvidia_ai_endpoints'
        # clients (ChatNVIDIA / NVIDIAEmbeddings) know how to handle NIM-specific request extensions that
        # the generic openai-compatible client doesn't:
        #   - nv-embedqa-e5-v5 is an "asymmetric" embedding model requiring input_type ("query" vs
        #     "passage") on every request; the openai provider has no idea this parameter exists and NIM
        #     rejects the request with a 400.
        #   - chat_template_kwargs (used to disable "thinking" mode on Nemotron) is a NIM-specific request
        #     field. LangChain's generic ChatOpenAI treats it as an arbitrary "model_kwargs" entry and
        #     forwards it verbatim into openai-python's AsyncCompletions.create(), which doesn't accept it
        #     as a real parameter and raises a TypeError. ChatNVIDIA routes it correctly via extra_body.
        "embedders": {
            "nv-embedqa-e5-v5": {
                "_type": "nim",
                "model_name": "nvidia/nvidia/nv-embedqa-e5-v5",
                "api_key": "${NVIDIA_API_KEY}",
                "base_url": NIM_BASE_URL,
            }
        },
        "memory": {
            "user_memory": {
                "_type": "redis_memory",
                "host": "localhost",
                "port": 6379,
                "embedder": "nv-embedqa-e5-v5",
            }
        },
        "functions": functions,
        "llms": {
            "nim_llm": {
                "_type": "nim",
                "model_name": "nvidia/nvidia/nemotron-3-nano-30b-a3b",
                "temperature": 0.0,
                "api_key": "${NVIDIA_API_KEY}",
                "base_url": NIM_BASE_URL,
                "chat_template_kwargs": {"enable_thinking": False},
            }
        },
        "workflow": {
            "_type": "react_agent",
            "tool_names": tool_names,
            "llm_name": "nim_llm",
            "verbose": True,
            "parse_agent_response_max_retries": 3,
            "additional_instructions": additional_instructions,
        },
    }


with open("memory_workflow.yml", "w") as f:
    yaml.safe_dump(build_memory_config(include_delete=False), f, sort_keys=False)

print("Wrote memory_workflow.yml")

### A known bug in this release: pre-create the Redis index ourselves

Checked directly in the installed source (`nat/plugins/redis/schema.py`, `ensure_index_exists`): it only treats a
RediSearch error containing `"no such index"` or `"Index needs recreation"` as "the index doesn't exist yet, go
create it." Newer Redis Stack images report a missing index as `"Unknown index name"` instead, which matches
neither string — so on a fresh Redis instance, this pinned `nvidia-nat-redis==1.8.0` plugin can treat completely
normal first-run initialization as a fatal error. Rather than patch the installed package (which would get
overwritten on any reinstall), we sidestep the buggy code path entirely: we create the index ourselves, using the
toolkit's own schema-building function so it's byte-for-byte what `redis_memory` expects, before `nat run` ever
calls `ensure_index_exists`. By the time it runs, the index already exists, so it takes the fast, non-buggy path.

In [ ]:
import redis.exceptions as redis_exceptions
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from redis.commands.search.indexDefinition import IndexDefinition, IndexType

from nat.plugins.redis.schema import INDEX_NAME, create_schema

# Measure the embedder's real output dimension using the same client class the toolkit uses
# internally (NVIDIAEmbeddings — it handles the input_type="query"/"passage" parameter that
# nv-embedqa-e5-v5 requires automatically), so the index we pre-create matches exactly. Respects
# NVIDIA_BASE_URL the same way the YAML config above does.
embed_client = NVIDIAEmbeddings(
    model="nvidia/nvidia/nv-embedqa-e5-v5",
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url=os.environ.get("NVIDIA_BASE_URL"),
)
test_embedding = embed_client.embed_query("test")
embedding_dim = len(test_embedding)
print(f"Embedder output dimension: {embedding_dim}")

r = redis.Redis(host="localhost", port=6379)
try:
    r.ft(INDEX_NAME).info()
    print(f"Redis index '{INDEX_NAME}' already exists — leaving it as is.")
except redis_exceptions.ResponseError:
    schema = create_schema(embedding_dim)
    r.ft(INDEX_NAME).create_index(schema, definition=IndexDefinition(prefix=["nat:"], index_type=IndexType.JSON))
    print(f"Created Redis index '{INDEX_NAME}' with embedding dimension {embedding_dim}.")

## 6. Teach it a fact, then recall it in a new process

We run `nat run` **twice**, as two entirely separate CLI invocations. If the second call can answer correctly, that
proves the fact was actually persisted in Redis — not just sitting in a conversation history we happened to keep
in memory in this notebook's process.

In [ ]:
!nat run --config_file memory_workflow.yml --user_id demo_user --input \
    "My user ID is demo_user. Please remember that I'm allergic to peanuts and my favorite programming language is Rust."

In [ ]:
!nat run --config_file memory_workflow.yml --user_id demo_user --input \
    "My user ID is demo_user. What food should I avoid, and what's my favorite programming language?"

If that second call correctly mentions peanuts and Rust, look at the verbose `Thought`/`Action` trace above it —
you should see the agent call `get_memory` with a query like *"user's allergies and favorite programming
language"* before producing the final answer, rather than guessing from nothing.

## 7. Clean up

`delete_memory` is a first-class tool, not an afterthought — memory is personal data, and users should be able to
ask an agent to forget them.

In [ ]:
with open("memory_workflow_with_delete.yml", "w") as f:
    yaml.safe_dump(build_memory_config(include_delete=True), f, sort_keys=False)

print("Wrote memory_workflow_with_delete.yml")

In [ ]:
!nat run --config_file memory_workflow_with_delete.yml --user_id demo_user --input \
    "My user ID is demo_user. Please delete everything you have stored about me."

In [ ]:
# Stop and remove the Redis container when you're done (safe to skip if you're using your own Redis Stack instance).
!docker stop nat-memory-redis && docker rm nat-memory-redis

## Wrap-up

- **Working memory** (this session's context window) vs. **long-term memory** (an external store the agent writes
  to and reads back from across sessions) are different mechanisms solving different problems — don't reach for a
  memory backend just to keep a conversation coherent; `max_history` already does that.
- The toolkit's memory feature is backend-agnostic: a `memory:` section names a client, and `add_memory` /
  `get_memory` / `delete_memory` are ordinary tools layered on top — the same pattern as any other tool from
  notebook 1.
- Everything is scoped by `user_id`, and that ID is an LLM-filled tool argument, not something injected for you —
  make sure your prompts (or `additional_instructions`) give the agent a stable identifier to use.
- We used Redis because it's the only shipped backend that doesn't require a third-party account; `mem0ai` /
  `zep-cloud` / `memmachine` follow the identical config pattern if you already have one of those set up.

See the toolkit's own [`examples/memory/redis`](https://github.com/NVIDIA/NeMo-Agent-Toolkit/tree/main/examples/memory/redis)
and [`examples/memory/memmachine`](https://github.com/NVIDIA/NeMo-Agent-Toolkit/tree/main/examples/memory/memmachine)
for more complete reference examples.